# yfinance exploration: what does Yahoo actually return for NSE stocks?

**Purpose:** before writing the ingestion code in Step 2, check the exact shape, labels, units and failure behaviour of the yfinance data we depend on. Every finding below drives a decision in `data/ingestion/`.

| | |
|---|---|
| **Tickers** | `TCS.NS` (IT services), `HDFCBANK.NS` (bank), `FAKEXYZ.NS` (does not exist) |
| **Library** | yfinance (version printed below); results can change between versions |
| **Data** | live from Yahoo, so the numbers shift with every run; the *structure* is what matters |

**Contents**
- A. Prices: column layout, `Adj Close`, index type
- B. Unknown ticker: how failure shows up
- C. Financials (TCS): statement layout, row labels, units
- D. The bank case (HDFC Bank): which labels go missing
- E. Stock master data: `info` keys
- Summary: implications for Step 2

In [1]:
import sys
import warnings

import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:,.2f}".format)

STOCK = "TCS.NS"
BANK = "HDFCBANK.NS"
FAKE = "FAKEXYZ.NS"

print(f"yfinance {yf.__version__} | pandas {pd.__version__}")
print(f"run on   {pd.Timestamp.now():%Y-%m-%d %H:%M}")
print(f"python   {sys.executable}")

yfinance 1.3.0 | pandas 3.0.3
run on   2026-11-07 16:30
python   /Users/bhanubhadouria/Library/Caches/pypoetry/virtualenvs/mosaic-drishti-mWWWkczO-py3.11/bin/python


---
## A. Prices

One month of daily prices, with `auto_adjust=False`. (The default, `auto_adjust=True`, overwrites Open/High/Low/Close with adjusted values and drops `Adj Close`, which loses the real traded close.)

In [2]:
df = yf.download(STOCK, period="1mo", auto_adjust=False, progress=False)
df.head()

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,TCS.NS,TCS.NS,TCS.NS,TCS.NS,TCS.NS,TCS.NS
Date,,,,,,
2026-10-06,"2,088.90","2,100.00","2,115.00","2,082.70","2,097.00",2382510
2026-10-07,"2,069.30","2,080.30","2,105.00","2,076.00","2,085.20",1349758
2026-10-08,"2,065.02","2,076.00","2,141.50","2,060.00","2,104.00",4288031
2026-10-09,"2,144.60","2,156.00","2,204.00","2,086.00","2,086.00",11692314
2026-10-12,"2,113.76","2,125.00","2,195.10","2,120.90","2,184.90",3540523


### Q1. Are the columns flat, or a MultiIndex?

In [3]:
print("type:   ", type(df.columns).__name__)
print("levels: ", df.columns.nlevels, list(df.columns.names))
print("level 0:", df.columns.get_level_values(0).tolist())
print("level 1:", df.columns.get_level_values(1).unique().tolist())

type:    MultiIndex
levels:  2 ['Price', 'Ticker']
level 0: ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
level 1: ['TCS.NS']


> **Finding:** a **2-level MultiIndex**. Level 0 (`Price`) is the field name, level 1 (`Ticker`) is the symbol, so every column is a pair like `('Close', 'TCS.NS')`. This happens even when only one ticker is requested.

### Q2. Is there an `Adj Close`, and how different is it from `Close`?

In [4]:
prices = pd.DataFrame({
    "Close": df["Close"][STOCK],
    "Adj Close": df["Adj Close"][STOCK],
})
prices["Diff"] = prices["Close"] - prices["Adj Close"]
prices["Diff %"] = prices["Diff"] / prices["Close"] * 100
prices.head(10)

,Close,Adj Close,Diff,Diff %
Date,,,,
2026-10-06,"2,100.00","2,088.90",11.10,0.53
2026-10-07,"2,080.30","2,069.30",11.00,0.53
2026-10-08,"2,076.00","2,065.02",10.98,0.53
2026-10-09,"2,156.00","2,144.60",11.40,0.53
2026-10-12,"2,125.00","2,113.76",11.24,0.53
2026-10-13,"2,269.30","2,257.30",12.00,0.53
2026-10-14,"2,304.90","2,304.90",0.00,0.00
2026-10-15,"2,350.00","2,350.00",0.00,0.00
2026-10-16,"2,465.00","2,465.00",0.00,0.00


In [5]:
# The step change in Diff lines up with a dividend: Yahoo back-adjusts every price before the ex-date
yf.Ticker(STOCK).dividends.tail(3)

Date
2026-05-25 09:15:00+05:30   31.00
2026-07-15 09:15:00+05:30   12.00
2026-10-14 09:15:00+05:30   12.00
Name: Dividends, dtype: float64

> **Finding:** yes, `Adj Close` exists when `auto_adjust=False`. Before the most recent ex-dividend date, `Adj Close` is lower than `Close` by roughly the dividend amount (about 0.5%); from the ex-date onwards the two are identical. `Adj Close` is **back-adjusted**: Yahoo rewrites *historical* values each time a dividend or split happens, so a stored `adjusted_close` goes stale and must be refreshed, not just appended.

### Q3. What is the index dtype, and does it have a timezone?

In [6]:
print("dtype:", df.index.dtype)
print("tz:   ", df.index.tz)
print("first:", df.index[:2].tolist())
print("dividends index (for contrast):", yf.Ticker(STOCK).dividends.index.dtype)

dtype: datetime64[s]
tz:    None
first: [Timestamp('2026-10-06 00:00:00'), Timestamp('2026-10-07 00:00:00')]
dividends index (for contrast): datetime64[s, Asia/Kolkata]


> **Finding:** a **timezone-naive** `datetime64` index at midnight (dates only, no time of day). It converts cleanly to `PriceHistory.date` with `df.index.date`. Note that other yfinance endpoints are *not* consistent: the dividends index is timezone-aware (`Asia/Kolkata`, 09:15), so never assume.

### Q4. Does `multi_level_index=False` flatten the columns?

In [7]:
df_flat = yf.download(STOCK, period="1mo", auto_adjust=False, progress=False, multi_level_index=False)

print("levels: ", df_flat.columns.nlevels)
print("columns:", df_flat.columns.tolist())
print("same values as the MultiIndex version:", (df_flat["Close"] == df["Close"][STOCK]).all())

levels:  1
columns: ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']
same values as the MultiIndex version: True


> **Finding:** yes. Columns become plain names (`Adj Close`, `Close`, `High`, `Low`, `Open`, `Volume`) with no ticker suffix, and the values are unchanged. Ingestion should download **one ticker at a time** with `auto_adjust=False, multi_level_index=False`.

---
## B. Unknown ticker

### Q5. Does a bad symbol raise, return an empty DataFrame, or just print something?

In [8]:
try:
    bad = yf.download(FAKE, period="1mo", auto_adjust=False, progress=False, multi_level_index=False)
    outcome = "no exception"
except Exception as e:
    bad, outcome = None, f"raised {type(e).__name__}: {e}"

print("outcome:", outcome)
if bad is not None:
    print("empty:  ", bad.empty)
    print("shape:  ", bad.shape)
    print("columns:", bad.columns.tolist())

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FAKEXYZ.NS"}}}
$FAKEXYZ.NS: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['FAKEXYZ.NS']: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


outcome: no exception
empty:   True
shape:   (0, 6)
columns: ['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']


> **Finding:** **no exception.** yfinance *logs* an error (the red text above: "possibly delisted; no price data found") and returns an **empty DataFrame** that still has the normal columns. The logged text is invisible to calling code, so Step 2 must check `df.empty` and raise `UnknownTickerError` itself.
>
> Caveat: an empty result can also mean "valid ticker, no trading in that window" (a holiday-only range, a suspended stock). Use a long enough period, or confirm with `info`, before treating empty as unknown.

---
## C. Financials (TCS)

In [9]:
STATEMENTS = ["financials", "quarterly_financials", "balance_sheet", "quarterly_balance_sheet"]


def statement_overview(ticker):
    """One row per statement: its shape and the period dates it covers."""
    t = yf.Ticker(ticker)
    rows = []
    for name in STATEMENTS:
        s = getattr(t, name)
        rows.append({
            "statement": name,
            "row labels": s.shape[0],
            "periods": s.shape[1],
            "newest": s.columns.max().date(),
            "oldest": s.columns.min().date(),
            "period dates": ", ".join(str(c.date()) for c in s.columns),
        })
    return pd.DataFrame(rows).set_index("statement")


tcs = yf.Ticker(STOCK)
tcs.financials.iloc[:5, :]

,2026-03-31,2025-03-31,2024-03-31,2023-03-31
Tax Effect Of Unusual Items,"-9,018,360,000.00","445,280,000.00","-1,912,980,950.69","277,560,000.00"
Tax Rate For Calcs,0.25,0.25,0.26,0.26
Normalized EBITDA,"759,400,000,000.00","711,930,000,000.00","685,060,000,000.00","626,000,000,000.00"
Total Unusual Items,"-36,660,000,000.00","1,760,000,000.00","-7,460,000,000.00","1,080,000,000.00"
Total Unusual Items Excluding Goodwill,"-36,660,000,000.00","1,760,000,000.00","-7,460,000,000.00","1,080,000,000.00"


### Q6–Q7. What are the rows and columns, and how many periods are there?

In [10]:
print("rows are:   ", tcs.financials.index.dtype, "(line-item labels)")
print("columns are:", tcs.financials.columns.dtype, "(period end dates, newest first)")
statement_overview(STOCK)

rows are:    str (line-item labels)
columns are: datetime64[s] (period end dates, newest first)


,row labels,periods,newest,oldest,period dates
statement,,,,,
financials,48,4,2026-03-31,2023-03-31,"2026-03-31, 2025-03-31, 2024-03-31, 2023-03-31"
quarterly_financials,48,5,2026-09-30,2025-06-30,"2026-09-30, 2026-06-30, 2026-03-31, 2025-12-31, 2025-06-30"
balance_sheet,81,5,2026-03-31,2022-03-31,"2026-03-31, 2025-03-31, 2024-03-31, 2023-03-31, 2022-03-31"
quarterly_balance_sheet,80,6,2026-09-30,2025-03-31,"2026-09-30, 2026-06-30, 2026-03-31, 2025-12-31, 2025-06-30, 2025-03-31"


> **Finding:** **rows are line items** (text labels) and **columns are period-end dates**, newest first. That's the transpose of what we store (one row per period), so ingestion will use `.T`.
>
> Annual statements cover about 4–5 years (financial years ending 31 March). Quarterly statements cover only about 5–6 quarters, and can have **gaps**: TCS's quarterly income statement skips 2025-09-30. The balance sheet's oldest period (2022-03-31) is entirely NaN, so drop all-NaN periods before inserting.

### Q8. The exact row label for each field we need

In [11]:
def find_labels(statement, keyword):
    """All row labels containing keyword (case-insensitive)."""
    return statement.index[statement.index.str.contains(keyword, case=False)].tolist()


for kw in ["revenue", "net income", "eps", "ebit"]:
    print(f"{kw!r:14} -> {find_labels(tcs.financials, kw)}")
print()
for kw in ["debt", "stockholders", "total assets", "current liabilities"]:
    print(f"{kw!r:22} -> {find_labels(tcs.balance_sheet, kw)}")

'revenue'      -> ['Reconciled Cost Of Revenue', 'Cost Of Revenue', 'Total Revenue', 'Operating Revenue']
'net income'   -> ['Net Income From Continuing Operation Net Minority Interest', 'Net Income From Continuing And Discontinued Operation', 'Net Income Common Stockholders', 'Net Income', 'Net Income Including Noncontrolling Interests', 'Net Income Continuous Operations']
'eps'          -> ['Diluted EPS', 'Basic EPS']
'ebit'         -> ['Normalized EBITDA', 'EBITDA', 'EBIT']

'debt'                 -> ['Total Debt', 'Long Term Debt And Capital Lease Obligation', 'Current Debt And Capital Lease Obligation']
'stockholders'         -> ['Stockholders Equity']
'total assets'         -> ['Total Assets']
'current liabilities'  -> ['Total Non Current Liabilities Net Minority Interest', 'Other Non Current Liabilities', 'Current Liabilities', 'Other Current Liabilities']


In [12]:
# The labels chosen for ingestion (exact strings)
INCOME_LABELS = {
    "revenue": "Total Revenue",
    "net_income": "Net Income",
    "basic_eps": "Basic EPS",
    "diluted_eps": "Diluted EPS",
    "ebit": "EBIT",
}
BALANCE_LABELS = {
    "total_debt": "Total Debt",
    "stockholders_equity": "Stockholders Equity",
    "total_assets": "Total Assets",
    "current_liabilities": "Current Liabilities",
}

# dropna removes periods with no data at all (the balance sheet's oldest year)
pd.concat([
    tcs.financials.loc[list(INCOME_LABELS.values())],
    tcs.balance_sheet.loc[list(BALANCE_LABELS.values())],
], sort=False).dropna(axis=1, how="all")

,2026-03-31,2025-03-31,2024-03-31,2023-03-31
Total Revenue,"2,670,210,000,000.00","2,553,240,000,000.00","2,408,930,000,000.00","2,254,580,000,000.00"
Net Income,"492,100,000,000.00","485,530,000,000.00","459,080,000,000.00","421,470,000,000.00"
Basic EPS,136.01,134.19,125.88,115.19
Diluted EPS,136.01,134.19,125.88,115.19
EBIT,"667,140,000,000.00","661,270,000,000.00","627,750,000,000.00","576,860,000,000.00"
Total Debt,"112,830,000,000.00","93,920,000,000.00","80,210,000,000.00","76,880,000,000.00"
Stockholders Equity,"1,072,400,000,000.00","947,560,000,000.00","904,890,000,000.00","904,240,000,000.00"
Total Assets,"1,823,720,000,000.00","1,596,290,000,000.00","1,464,490,000,000.00","1,436,510,000,000.00"
Current Liabilities,"609,140,000,000.00","530,010,000,000.00","461,040,000,000.00","435,580,000,000.00"


> **Finding:** the exact labels are `Total Revenue`, `Net Income`, `Basic EPS`, `Diluted EPS`, `EBIT`, `Total Debt`, `Stockholders Equity`, `Total Assets` and `Current Liabilities`.
>
> Watch out for near-duplicates: `Operating Revenue` sits next to `Total Revenue`; `Net Income Common Stockholders` and several other "Net Income ..." variants exist; `Common Stock Equity` sits next to `Stockholders Equity`. Ingestion should look labels up by these exact strings, and treat a missing label as `None`, never a `KeyError`.

### Q9. Units: rupees, lakhs or crores?

Sanity check: TCS's annual revenue is roughly ₹2.5 lakh crore (₹2,50,000 crore).

In [13]:
CRORE = 1e7  # 1 crore = 10,000,000 rupees

revenue = tcs.financials.loc["Total Revenue"]
pd.DataFrame({
    "raw value": revenue,
    "in ₹ crore": revenue / CRORE,
    "in ₹ lakh crore": revenue / CRORE / 1e5,
})

,raw value,in ₹ crore,in ₹ lakh crore
2026-03-31,"2,670,210,000,000.00","267,021.00",2.67
2025-03-31,"2,553,240,000,000.00","255,324.00",2.55
2024-03-31,"2,408,930,000,000.00","240,893.00",2.41
2023-03-31,"2,254,580,000,000.00","225,458.00",2.25


> **Finding:** values are in **plain rupees**. Divided by 10⁷, revenue is about ₹2.5–2.7 lakh crore, which matches. Store them as-is (`Numeric(20, 2)` holds them comfortably) and convert to crores only for display. EPS is in rupees per share.

---
## D. The bank case (HDFC Bank)

### Q10. Which of the Q8 labels are missing or NaN for a bank?

In [14]:
statement_overview(BANK)

,row labels,periods,newest,oldest,period dates
statement,,,,,
financials,41,4,2026-03-31,2023-03-31,"2026-03-31, 2025-03-31, 2024-03-31, 2023-03-31"
quarterly_financials,31,2,2025-06-30,2025-03-31,"2025-06-30, 2025-03-31"
balance_sheet,59,4,2026-03-31,2023-03-31,"2026-03-31, 2025-03-31, 2024-03-31, 2023-03-31"
quarterly_balance_sheet,59,4,2026-03-31,2025-03-31,"2026-03-31, 2025-09-30, 2025-06-30, 2025-03-31"


In [15]:
def label_coverage(ticker):
    """For every label we need: 'missing' if the row doesn't exist, else 'non-null / total periods'."""
    t = yf.Ticker(ticker)
    sources = [  # (labels, annual statement, quarterly statement)
        (INCOME_LABELS, t.financials, t.quarterly_financials),
        (BALANCE_LABELS, t.balance_sheet, t.quarterly_balance_sheet),
    ]
    rows = {}
    for labels, annual, quarterly in sources:
        for label in labels.values():
            rows[label] = {
                period: "missing" if label not in s.index else f"{s.loc[label].notna().sum()} / {s.shape[1]}"
                for period, s in (("annual", annual), ("quarterly", quarterly))
            }
    return pd.DataFrame(rows).T


pd.concat({STOCK: label_coverage(STOCK), BANK: label_coverage(BANK)}, axis=1)

TCS.NS           HDFCBANK.NS          
                    annual quarterly      annual quarterly
Total Revenue        4 / 4     5 / 5       4 / 4     2 / 2
Net Income           4 / 4     5 / 5       4 / 4     2 / 2
Basic EPS            4 / 4     5 / 5       4 / 4     1 / 2
Diluted EPS          4 / 4     5 / 5       4 / 4     1 / 2
EBIT                 4 / 4     5 / 5     missing   missing
Total Debt           4 / 5     5 / 6       4 / 4     4 / 4
Stockholders Equity  4 / 5     5 / 6       4 / 4     4 / 4
Total Assets         4 / 5     5 / 6       4 / 4     4 / 4
Current Liabilities  4 / 5     5 / 6     missing   missing

In [16]:
# EPS history for the bank: basic and diluted disagree by about 2x in some years
yf.Ticker(BANK).financials.loc[["Net Income", "Basic EPS", "Diluted EPS"]]

,2026-03-31,2025-03-31,2024-03-31,2023-03-31
Net Income,"704,793,400,000.00","673,508,300,000.00","622,656,600,000.00","495,446,900,000.00"
Basic EPS,45.89,45.01,88.66,44.51
Diluted EPS,45.75,44.82,44.16,88.68


> **Finding:** for HDFC Bank, **`EBIT` and `Current Liabilities` don't exist at all**. Banks don't report operating profit or a current/non-current split. Revenue, net income, EPS, total debt, equity and assets are present. Ratios built on the missing fields (ROCE, current ratio) can't be computed for banks and must stay `NULL`.
>
> Two more bank-specific problems:
> - **Quarterly data is thin and stale.** Only a couple of quarters come back, older than TCS's, and quarterly EPS is partly NaN.
> - **EPS is inconsistently restated.** Net income grows smoothly, but in FY2024 basic EPS is about double diluted EPS, and in FY2023 it's the other way round. That pattern fits a 1:1 bonus issue where Yahoo restated some older figures but not others. Basic and diluted EPS should never differ by 2×, so ingestion should flag that case, and agents shouldn't compare EPS across years without checking.

---
## E. Stock master data

### Q11. Which `info` keys give the company name, sector and industry?

In [17]:
WANTED = ["longName", "shortName", "sector", "sectorKey", "industry", "industryKey",
          "fullExchangeName", "exchange", "currency", "financialCurrency"]

infos = {tk: yf.Ticker(tk).info for tk in (STOCK, BANK)}
pd.DataFrame({tk: {k: info.get(k) for k in WANTED} for tk, info in infos.items()})

,TCS.NS,HDFCBANK.NS
longName,Tata Consultancy Services Limited,HDFC Bank Limited
shortName,TATA CONSULTANCY SERV LT,HDFC BANK LTD
sector,Technology,Financial Services
sectorKey,technology,financial-services
industry,Information Technology Services,Banks - Regional
industryKey,information-technology-services,banks-regional
fullExchangeName,NSE,NSE
exchange,NSI,NSI
currency,INR,INR
financialCurrency,INR,INR


> **Finding:**
> - **Name:** `longName` (e.g. "Tata Consultancy Services Limited"). `shortName` is a truncated, all-caps ticker-tape version; don't use it.
> - **Sector:** `sector` (display text). `sectorKey` is the slug form.
> - **Industry:** `industry`. `industryKey` is the slug form.
> - **Exchange:** use `fullExchangeName` (`NSE`); `exchange` is Yahoo's internal code (`NSI`), which doesn't fit our `"NSE" | "BSE"` convention.
>
> `info` is a plain dict with no guaranteed keys, so read with `.get()`.

---
## Summary: implications for Step 2

| Topic | Decision for ingestion |
|---|---|
| Download call | `yf.download(symbol, auto_adjust=False, multi_level_index=False, progress=False)`, one ticker at a time |
| Price columns | `Close` → `close`, `Adj Close` → `adjusted_close`, `Open` → `open_price` |
| Dates | index is timezone-naive midnight; convert with `.date` |
| Adjusted prices | back-adjusted by Yahoo after dividends/splits, so re-fetch and **upsert** history, don't only append |
| Unknown ticker | no exception, just an empty DataFrame and a log line; check `df.empty` and raise `UnknownTickerError` |
| Statement layout | rows = labels, columns = period dates; transpose with `.T`, drop all-NaN periods |
| Labels | exact strings from Q8; a missing label → `None`, never `KeyError` |
| Units | plain rupees; store as-is |
| Banks | `EBIT` and `Current Liabilities` absent; quarterly data sparse; EPS not restated across bonus issues |
| Master data | `longName`, `sector`, `industry`, `fullExchangeName`, read with `.get()` |

---
## F. Checking `validate_prices()` (Step 4)

Run the real validator on 5 years of TCS, then break rows on purpose to confirm each rule fires with the right reason, and that too many failures reject the whole ticker.

In [18]:
import logging
import sys
from pathlib import Path

import numpy as np

# Make the project importable: walk up from this notebook to the folder with pyproject.toml
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from config.settings import settings
from data.ingestion.exceptions import DataQualityError
from data.ingestion.price_ingestion import fetch_prices, validate_prices

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s", force=True)
print("MAX_REJECT_RATIO:", settings.MAX_REJECT_RATIO)

MAX_REJECT_RATIO: 0.05


### F1. Clean data: 5 years of TCS

In [19]:
prices = fetch_prices(STOCK)  # period defaults to settings.MAX_HISTORY_YEARS
valid, rejected = validate_prices(STOCK, prices)

print(f"fetched:  {len(prices)}")
print(f"valid:    {len(valid)}")
print(f"rejected: {len(rejected)}")
print(f"date range: {prices['date'].min()} to {prices['date'].max()}")

fetched:  1237
valid:    1237
rejected: 0
date range: 2021-11-08 to 2026-11-06


> **Finding:** no rows rejected over 5 years of real TCS data, so the rules (with the 0.01 tolerance) don't flag normal Yahoo noise.

### F2. Three rows broken on purpose

One NaN close, one high below low, one duplicated date, plus a row that breaks several rules at once to show that every reason is listed.

In [20]:
broken = prices.copy()
broken.loc[10, "close"] = np.nan                                 # -> missing_price
broken.loc[20, "high"] = broken.loc[20, "low"] - 5                # -> high_below_others
broken.loc[30, "date"] = broken.loc[29, "date"]                   # -> duplicate_date (row 29 is kept)
broken.loc[40, ["high", "close", "volume"]] = [-1.0, 0.0, -5]     # -> several rules at once

valid, rejected = validate_prices(STOCK, broken)
print(f"valid: {len(valid)}, rejected: {len(rejected)}")
rejected[["date", "open_price", "high", "low", "close", "volume", "reason"]]

WARNING TCS.NS 2021-11-23 rejected: missing_price
WARNING TCS.NS 2021-12-07 rejected: high_below_others
WARNING TCS.NS 2021-12-20 rejected: duplicate_date
WARNING TCS.NS 2022-01-04 rejected: non_positive_price; high_below_others; low_above_others; negative_volume


valid: 1233, rejected: 4


,date,open_price,high,low,close,volume,reason
10,2021-11-23,"3,445.00","3,480.95","3,407.80",NaN,2153461,missing_price
20,2021-12-07,"3,562.50","3,519.30","3,524.30","3,584.50",1484458,high_below_others
30,2021-12-20,"3,582.65","3,641.20","3,566.40","3,608.30",2340989,duplicate_date
40,2022-01-04,"3,831.10",-1.00,"3,811.70",0.00,-5,non_positive_price; high_below_others; low_above_others; negative_volume


> **Finding:** each broken row is rejected with the expected reason, and the WARNING log lines above name the symbol, date and reason. Row 29 (the first copy of the duplicated date) stays valid. Row 40 lists all four rules it broke. The NaN close does *not* also trip `low_above_others`, because pandas' `min`/`max` skip NaN; only `missing_price` catches it, which is why that rule exists.

### F3. Too many failures reject the whole ticker

In [21]:
logging.getLogger("data.ingestion.price_ingestion").setLevel(logging.ERROR)  # hide ~120 row warnings

too_broken = prices.copy()
n_bad = int(len(too_broken) * 0.10)
too_broken.loc[too_broken.index[:n_bad], "volume"] = -1

try:
    validate_prices(STOCK, too_broken)
    print("NOT raised (unexpected)")
except DataQualityError as e:
    print(f"raised:  {type(e).__name__}: {e}")
    print(f"counts:  failed={e.failed}, total={e.total}, ratio={e.failed / e.total:.1%}")

at_limit = prices.copy()
n_limit = int(len(at_limit) * settings.MAX_REJECT_RATIO)
at_limit.loc[at_limit.index[:n_limit], "volume"] = -1
_, rejected = validate_prices(STOCK, at_limit)
print(f"at limit: {len(rejected)}/{len(at_limit)} = {len(rejected) / len(at_limit):.1%} rejected, no exception")

logging.getLogger("data.ingestion.price_ingestion").setLevel(logging.NOTSET)

raised:  DataQualityError: TCS.NS: 123/1237 rows failed validation
counts:  failed=123, total=1237, ratio=9.9%
at limit: 61/1237 = 4.9% rejected, no exception


> **Finding:** breaking 10% of rows raises `DataQualityError` with the exact counts, which is what the Step 9 summary will print. A rejection rate just under the 5% limit is allowed through: the rule is "more than 5%", not "5% or more".

---
## G. Checking `upsert_stock()` (Step 5)

Fetch first (network, no transaction open), then upsert inside a short transaction. Run it twice. Expected: the same id both times, `created_at` unchanged, `updated_at` moved forward.

In [22]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from sqlalchemy import text

from data.database import SessionLocal
from data.ingestion.stocks import fetch_stock_info, upsert_stock

ROW_SQL = text("SELECT id, symbol, name, sector, industry, exchange, created_at, updated_at FROM stocks WHERE symbol = :s")


def show_row(session, symbol=STOCK):
    return pd.DataFrame([session.execute(ROW_SQL, {"s": symbol}).mappings().one()])


fetch_stock_info(STOCK)

{'name': 'Tata Consultancy Services',
 'sector': 'Technology',
 'industry': 'Information Technology Services',
 'exchange': 'NSE'}

### G1. First run

In [23]:
info = fetch_stock_info(STOCK)  # network, outside the transaction
with SessionLocal() as s:
    print("id:", upsert_stock(s, STOCK, info))
    s.commit()
    first = show_row(s)
first

id: 1


,id,symbol,name,sector,industry,exchange,created_at,updated_at
0,1,TCS.NS,Tata Consultancy Services,Technology,Information Technology Services,NSE,2026-11-07 13:20:11.914997+00:00,2026-11-07 13:30:15.266280+00:00


### G2. Second run

In [24]:
info = fetch_stock_info(STOCK)
with SessionLocal() as s:
    print("id:", upsert_stock(s, STOCK, info))
    s.commit()
    second = show_row(s)

pd.DataFrame({
    "same id": [first.at[0, "id"] == second.at[0, "id"]],
    "created_at unchanged": [first.at[0, "created_at"] == second.at[0, "created_at"]],
    "updated_at moved forward": [second.at[0, "updated_at"] > first.at[0, "updated_at"]],
})

id: 1


,same id,created_at unchanged,updated_at moved forward
0,True,True,True


> **Finding:** the second run hit `ON CONFLICT DO UPDATE`: same id, `created_at` untouched, `updated_at` refreshed. The explicit `updated_at=func.now()` in `set_` is what moves it; the model's `onupdate` alone would not.

### G3. A failed `info` lookup must not erase stored sector/industry

Simulate `t.info` raising, upsert, and look at the row inside the same transaction, then roll back so the database is left as it was.

In [25]:
from unittest import mock

import yfinance as yf

with mock.patch.object(yf.Ticker, "info", new_callable=mock.PropertyMock, side_effect=RuntimeError("simulated outage")):
    failed_info = fetch_stock_info(STOCK)
print("info sent:", failed_info)

with SessionLocal() as s:
    upsert_stock(s, STOCK, failed_info)  # no mocking needed: upsert makes no network calls
    during = show_row(s)
    s.rollback()

during[["name", "sector", "industry"]]

TCS.NS: info lookup failed (simulated outage); sector/industry unknown


info sent: {'name': 'Tata Consultancy Services', 'sector': None, 'industry': None, 'exchange': 'NSE'}


,name,sector,industry
0,Tata Consultancy Services,Technology,Information Technology Services


> **Finding:** the upsert sent `sector=None, industry=None`, but the row still says Technology / Information Technology Services, thanks to `COALESCE(excluded.sector, stocks.sector)`. `name` is always the universe name, so it doesn't change whether or not Yahoo answered. And since `upsert_stock` no longer fetches, testing it needed no mocking at all; only the fetch was mocked.

---
## H. Checking `store_prices()` (Step 6)

The full pipeline for one ticker: every network call first, then one short transaction that upserts the stock and bulk-upserts its prices. Then the same cell again, to prove it's idempotent.

In [26]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from sqlalchemy import text

from data.database import SessionLocal
from data.ingestion.price_ingestion import fetch_prices, store_prices, validate_prices
from data.ingestion.stocks import fetch_stock_info, upsert_stock


def db_checks(stock_id):
    """The three psql checks from the step, run through SQLAlchemy."""
    with SessionLocal() as s:
        count = s.execute(text("SELECT count(*) FROM price_history WHERE stock_id = :id"), {"id": stock_id}).scalar_one()
        nan_closes = s.execute(text("SELECT count(*) FROM price_history WHERE close = 'NaN'::numeric")).scalar_one()
        latest = pd.read_sql(
            text("SELECT date, close, adjusted_close FROM price_history ORDER BY date DESC LIMIT 3"), s.connection()
        )
    print(f"rows for stock_id={stock_id}: {count}")
    print(f"NaN closes: {nan_closes}")
    return latest

### H1. First run

In [27]:
symbol = "TCS.NS"
df = fetch_prices(symbol)
valid, rejected = validate_prices(symbol, df)
info = fetch_stock_info(symbol)                  # every network call before the session opens

with SessionLocal() as s:
    stock_id = upsert_stock(s, symbol, info)
    written = store_prices(s, stock_id, valid)
    s.commit()

print(len(df), len(rejected), written)           # this is your TCS baseline

1237 0 1237


In [28]:
first_count_check = db_checks(stock_id)
first_count_check

rows for stock_id=1: 1237
NaN closes: 0


,date,close,adjusted_close
0,2026-11-06,"2,498.50","2,498.50"
1,2026-11-05,"2,495.70","2,495.70"
2,2026-11-04,"2,436.00","2,436.00"


### H2. Second run (same cell, unchanged)

In [29]:
symbol = "TCS.NS"
df = fetch_prices(symbol)
valid, rejected = validate_prices(symbol, df)
info = fetch_stock_info(symbol)                  # every network call before the session opens

with SessionLocal() as s:
    stock_id = upsert_stock(s, symbol, info)
    written = store_prices(s, stock_id, valid)
    s.commit()

print(len(df), len(rejected), written)           # this is your TCS baseline

1237 0 1237


In [30]:
db_checks(stock_id)

rows for stock_id=1: 1237
NaN closes: 0


,date,close,adjusted_close
0,2026-11-06,"2,498.50","2,498.50"
1,2026-11-05,"2,495.70","2,495.70"
2,2026-11-04,"2,436.00","2,436.00"


> **Finding:** the second run wrote the same number of rows, but the table's row count didn't change: every row hit `ON CONFLICT (stock_id, date) DO UPDATE` instead of inserting a duplicate. The pipeline is **idempotent**, so it's safe to re-run after a failure or on a schedule. No `NaN` reached the `NUMERIC` columns, and prices are stored rounded to 4 decimal places by `Numeric(14, 4)`.

---
## I. Checking `fetch_financials()` (Step 7b)

Annual and quarterly statements reshaped to one row per period, for an IT company and a bank.

In [31]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from data.ingestion.financials_ingestion import VALUE_COLUMNS, fetch_financials

fin = {sym: fetch_financials(sym) for sym in (STOCK, BANK)}

pd.DataFrame({
    sym: {
        "annual rows": (df.period_type == "annual").sum(),
        "quarterly rows": (df.period_type == "quarterly").sum(),
        "all value columns float64": (df[VALUE_COLUMNS].dtypes == "float64").all(),
        "all-NaN columns": ", ".join(c for c in VALUE_COLUMNS if df[c].isna().all()) or "none",
    }
    for sym, df in fin.items()
})

,TCS.NS,HDFCBANK.NS
annual rows,4,4
quarterly rows,5,4
all value columns float64,True,True
all-NaN columns,none,"ebit, current_liabilities"


### I1. HDFC Bank, annual rows

In [32]:
hdfc = fin[BANK]
hdfc[hdfc.period_type == "annual"]

,period_type,period_date,revenue,net_income,eps_basic,eps_diluted,ebit,total_debt,stockholders_equity,total_assets,current_liabilities
0,annual,2023-03-31,"1,205,375,000,000.00","495,446,900,000.00",44.51,88.68,NaN,"3,151,792,800,000.00","2,912,200,700,000.00","25,755,624,000,000.00",NaN
1,annual,2024-03-31,"1,986,934,300,000.00","622,656,600,000.00",88.66,44.16,NaN,"8,093,552,100,000.00","6,947,327,400,000.00","44,118,573,000,000.00",NaN
2,annual,2025-03-31,"2,371,508,200,000.00","673,508,300,000.00",45.01,44.82,NaN,"7,326,106,100,000.00","7,676,891,400,000.00","48,187,671,100,000.00",NaN
3,annual,2026-03-31,"2,495,617,500,000.00","704,793,400,000.00",45.89,45.75,NaN,"6,643,640,400,000.00","8,167,395,600,000.00","52,621,193,200,000.00",NaN


### I2. Checks

In [33]:
from datetime import date

tcs_fin = fin[STOCK]
print("HDFC ebit all NaN:               ", hdfc["ebit"].isna().all())
print("HDFC current_liabilities all NaN:", hdfc["current_liabilities"].isna().all())
print("TCS has a 2022-03-31 row:        ", (tcs_fin.period_date == date(2022, 3, 31)).any())
print()
print("TCS quarterly dates: ", [str(d) for d in tcs_fin[tcs_fin.period_type == "quarterly"].period_date])
print("HDFC quarterly dates:", [str(d) for d in hdfc[hdfc.period_type == "quarterly"].period_date])

HDFC ebit all NaN:                True
HDFC current_liabilities all NaN: True
TCS has a 2022-03-31 row:         False

TCS quarterly dates:  ['2025-06-30', '2025-12-31', '2026-03-31', '2026-06-30', '2026-09-30']
HDFC quarterly dates: ['2025-03-31', '2025-06-30', '2025-09-30', '2026-03-31']


> **Finding:** no crash for the bank: `ebit` and `current_liabilities` come back as all-NaN columns, because `reindex` turns an unreported label into NaN. TCS's empty 2022 balance-sheet year is dropped. The outer join keeps periods only one statement covers: HDFC's quarterly income statement has just 2 quarters, but its balance sheet adds 2 more, giving 4 quarterly rows (income columns NaN in the extra two).

---
## J. Checking `validate_financials()` (Step 7c)

Field-level rejection: an impossible value is nulled, the rest of the period is kept.

In [34]:
import logging
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from data.ingestion.financials_ingestion import fetch_financials, validate_financials

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s", force=True)
pd.set_option("display.max_colwidth", None)

raw_fin = {sym: fetch_financials(sym) for sym in (STOCK, BANK)}

### J1. HDFC Bank

In [35]:
hdfc_valid, hdfc_rejections = validate_financials(BANK, raw_fin[BANK])
hdfc_rejections

WARNING HDFCBANK.NS annual 2023-03-31 rejected (eps_mismatch): {'eps_basic': 44.51, 'eps_diluted': 88.68}
WARNING HDFCBANK.NS annual 2024-03-31 rejected (eps_mismatch): {'eps_basic': 88.66, 'eps_diluted': 44.16}


,period_type,period_date,rule,raw_values
0,annual,2023-03-31,eps_mismatch,"{'eps_basic': 44.51, 'eps_diluted': 88.68}"
1,annual,2024-03-31,eps_mismatch,"{'eps_basic': 88.66, 'eps_diluted': 44.16}"


In [36]:
# eps is NULL for the two bad years; revenue and net income are untouched
hdfc_valid[hdfc_valid.period_type == "annual"][["period_date", "revenue", "net_income", "eps"]]

,period_date,revenue,net_income,eps
0,2023-03-31,"1,205,375,000,000.00","495,446,900,000.00",NaN
1,2024-03-31,"1,986,934,300,000.00","622,656,600,000.00",NaN
2,2025-03-31,"2,371,508,200,000.00","673,508,300,000.00",44.82
3,2026-03-31,"2,495,617,500,000.00","704,793,400,000.00",45.75


### J2. TCS

In [37]:
tcs_valid, tcs_rejections = validate_financials(STOCK, raw_fin[STOCK])
print(f"rejections: {len(tcs_rejections)}, valid periods: {len(tcs_valid)}/{len(raw_fin[STOCK])}")
tcs_rejections

rejections: 0, valid periods: 9/9


,period_type,period_date,rule,raw_values


### J3. A loss-making period must not be rejected

In [38]:
loss = raw_fin[STOCK].iloc[[0]].copy()
loss[["eps_basic", "eps_diluted"]] = -10.0

loss_valid, loss_rejections = validate_financials(STOCK, loss)
print(f"rejections: {len(loss_rejections)}, eps kept: {loss_valid['eps'].tolist()}")

rejections: 0, eps kept: [-10.0]


> **Finding:** HDFC's two inconsistent EPS years are caught by `eps_mismatch` and nulled, with the raw figures kept in `rejections`; revenue and net income for those years survive. TCS has no rejections. A loss (basic = diluted = −10) passes, because the EPS rule only applies when basic EPS is positive.

---
## K. Checking `store_financials()` (Step 7d)

Fetch, validate and store financials for both stocks; run it twice to prove it's idempotent.

In [39]:
import logging
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from sqlalchemy import text

from data.database import SessionLocal
from data.ingestion.financials_ingestion import fetch_financials, store_financials, validate_financials
from data.ingestion.stocks import fetch_stock_info, upsert_stock

logging.basicConfig(level=logging.ERROR, force=True)  # the EPS warnings were shown in section J

### K1. First run

In [40]:
for symbol in ["TCS.NS", "HDFCBANK.NS"]:
    info = fetch_stock_info(symbol)
    valid, rejections = validate_financials(symbol, fetch_financials(symbol))   # all network first
    with SessionLocal() as s:
        stock_id = upsert_stock(s, symbol, info)
        print(symbol, store_financials(s, stock_id, valid))
        s.commit()

TCS.NS 9
HDFCBANK.NS 8


### K2. Second run

In [41]:
for symbol in ["TCS.NS", "HDFCBANK.NS"]:
    info = fetch_stock_info(symbol)
    valid, rejections = validate_financials(symbol, fetch_financials(symbol))   # all network first
    with SessionLocal() as s:
        stock_id = upsert_stock(s, symbol, info)
        print(symbol, store_financials(s, stock_id, valid))
        s.commit()

TCS.NS 9
HDFCBANK.NS 8


In [42]:
with SessionLocal() as s:
    counts = pd.read_sql(text(
        "SELECT s.symbol, count(*) FROM financial_data f JOIN stocks s ON s.id = f.stock_id GROUP BY s.symbol ORDER BY s.symbol"
    ), s.connection())
    annual = pd.read_sql(text(
        "SELECT s.symbol, f.period_date, f.eps, f.revenue FROM financial_data f JOIN stocks s ON s.id = f.stock_id "
        "WHERE f.period_type = 'annual' ORDER BY s.symbol, f.period_date"
    ), s.connection())
counts

,symbol,count
0,HDFCBANK.NS,8
1,TCS.NS,9


In [43]:
annual

,symbol,period_date,eps,revenue
0,HDFCBANK.NS,2023-03-31,NaN,"1,205,375,000,000.00"
1,HDFCBANK.NS,2024-03-31,NaN,"1,986,934,300,000.00"
2,HDFCBANK.NS,2025-03-31,44.82,"2,371,508,200,000.00"
3,HDFCBANK.NS,2026-03-31,45.75,"2,495,617,500,000.00"
4,TCS.NS,2023-03-31,115.19,"2,254,580,000,000.00"
5,TCS.NS,2024-03-31,125.88,"2,408,930,000,000.00"
6,TCS.NS,2025-03-31,134.19,"2,553,240,000,000.00"
7,TCS.NS,2026-03-31,136.01,"2,670,210,000,000.00"


> **Finding:** the second run wrote the same rows again but the counts didn't change (TCS 9, HDFC 8), so the upsert is idempotent. HDFC's eps is NULL for FY2023 and FY2024, exactly the two years `eps_mismatch` rejected. The insert also succeeded on the five columns added by migration `8ec34c228497`.

---
## L. Checking `compute_ratios()` (Step 8a)

A pure function: DataFrames in, DataFrame out. First hand-made numbers whose answers are known in advance, then real data.

In [44]:
import sys
from datetime import date
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from data.ingestion.ratios import RATIO_COLUMNS, compute_ratios

pd.set_option("display.float_format", "{:,.4f}".format)

BASE = dict(ebit=150.0, total_debt=100.0, total_assets=1000.0, current_liabilities=200.0)


def period(id_, period_date, eps, net_income=100.0, equity=500.0):
    return dict(id=id_, period_date=period_date, net_income=net_income, eps=eps, stockholders_equity=equity, **BASE)

### L1. Hand-made test

Net income 100, equity 500, EPS 10, price 200. By hand: ROE = 100/500 = **0.20**; P/E = 200/10 = **20**; P/B = P/E × ROE = **4.0** (check: book value per share = 500 / (100/10) = 50, and 200/50 = 4). Also D/E = 100/500 = 0.20 and ROCE = 150/(1000−200) = 0.1875.

In [45]:
fin = pd.DataFrame([period(1, date(2026, 3, 31), eps=10.0)])
prices = pd.DataFrame({"date": [date(2026, 3, 31)], "close": [200.0]})
compute_ratios(fin, prices, is_financial=False)

,id,debt_to_equity,roe,roce,pe_ratio,pb_ratio
0,1,0.2000,0.2000,0.1875,20.0000,4.0000


### L2. Share-basis check

Same net income both years, but the older year's EPS is 20 instead of 10, so its implied share count is 100/20 = 5 against the newer year's 10: half. The older P/E (and so P/B) must be NULL; the other ratios don't depend on EPS and stay.

In [46]:
fin = pd.DataFrame([period(1, date(2025, 3, 31), eps=20.0), period(2, date(2026, 3, 31), eps=10.0)])
prices = pd.DataFrame({"date": [date(2025, 3, 28), date(2026, 3, 31)], "close": [200.0, 200.0]})
compute_ratios(fin, prices, is_financial=False)

,id,debt_to_equity,roe,roce,pe_ratio,pb_ratio
0,1,0.2000,0.2000,0.1875,NaN,NaN
1,2,0.2000,0.2000,0.1875,20.0000,4.0000


In [47]:
# Unknown sector (info fetch failed) is treated like financial: D/E and ROCE are skipped
compute_ratios(fin, prices, is_financial=None)

,id,debt_to_equity,roe,roce,pe_ratio,pb_ratio
0,1,NaN,0.2000,NaN,NaN,NaN
1,2,NaN,0.2000,NaN,20.0000,4.0000


### L3. Real data: TCS and HDFC Bank

HDFC's prices weren't stored yet (only TCS's were, in section H), so the first cell runs the existing price pipeline for it. `is_financial` comes from the stored sector.

In [48]:
from sqlalchemy import text

from data.database import SessionLocal, engine
from data.ingestion.price_ingestion import fetch_prices, store_prices, validate_prices
from data.ingestion.stocks import fetch_stock_info, upsert_stock

FINANCIAL_SECTORS = {"Financial Services"}

df = fetch_prices(BANK)
valid, rejected = validate_prices(BANK, df)
info = fetch_stock_info(BANK)
with SessionLocal() as s:
    stock_id = upsert_stock(s, BANK, info)
    print(BANK, "prices:", len(df), "fetched,", len(rejected), "rejected,", store_prices(s, stock_id, valid), "written")
    s.commit()

HDFCBANK.NS prices: 1237 fetched, 0 rejected, 1237 written


In [49]:
def real_ratios(symbol):
    """Read one stock's annual financials and prices from the database and compute ratios."""
    with engine.connect() as c:
        stock = c.execute(text("SELECT id, sector FROM stocks WHERE symbol = :s"), {"s": symbol}).one()
        fin = pd.read_sql(
            text("SELECT * FROM financial_data WHERE stock_id = :id AND period_type = 'annual' ORDER BY period_date"),
            c, params={"id": stock.id},
        )
        prices = pd.read_sql(text("SELECT date, close FROM price_history WHERE stock_id = :id"), c, params={"id": stock.id})

    ratios = compute_ratios(fin, prices, is_financial=stock.sector in FINANCIAL_SECTORS if stock.sector else None)
    shown = fin[["id", "period_date", "eps"]].assign(
        implied_shares_bn=fin.net_income.astype(float) / fin.eps.astype(float) / 1e9
    )
    print(f"{symbol} (sector: {stock.sector})")
    return shown.merge(ratios, on="id").drop(columns="id")


real_ratios(STOCK)

TCS.NS (sector: Technology)


,period_date,eps,implied_shares_bn,debt_to_equity,roe,roce,pe_ratio,pb_ratio
0,2023-03-31,115.1900,3.6589,0.0850,0.4661,0.5763,27.8314,12.9723
1,2024-03-31,125.8800,3.6470,0.0886,0.5073,0.6256,30.7936,15.6226
2,2025-03-31,134.1900,3.6182,0.0991,0.5124,0.6202,26.8735,13.7700
3,2026-03-31,136.0100,3.6181,0.1052,0.4589,0.5493,17.3436,7.9586


In [50]:
real_ratios(BANK)

HDFCBANK.NS (sector: Financial Services)


,period_date,eps,implied_shares_bn,debt_to_equity,roe,roce,pe_ratio,pb_ratio
0,2023-03-31,NaN,NaN,NaN,0.1701,NaN,NaN,NaN
1,2024-03-31,NaN,NaN,NaN,0.0896,NaN,NaN,NaN
2,2025-03-31,44.8200,15.0270,NaN,0.0877,NaN,20.3949,1.7893
3,2026-03-31,45.7500,15.4053,NaN,0.0863,NaN,15.9902,1.3798


> **Findings:**
> - **The hand-made test and the share-basis test give exactly the expected values.** An unknown sector skips D/E and ROCE.
> - **TCS:**
>   - ROE of 46–51% and ROCE of 55–63% are very high but typical of an asset-light IT company.
>   - D/E of about 0.1 is lease obligations: TCS has essentially no borrowings, and yfinance counts leases as "Total Debt".
>   - The implied share count is stable (3.62–3.66 bn), so every P/E passes the share-basis check.
>   - **Surprise:** P/E drops from 26.9 to 17.3 in FY2026. EPS barely changed, but the 31 March close fell from ₹3,606 to ₹2,359. The share count is unchanged, so this isn't a share-basis artefact. It's what the price data says, and worth confirming against an outside price source before an agent quotes it.
> - **HDFC Bank:**
>   - D/E and ROCE are NULL (financial sector), as intended.
>   - P/E is NULL for FY2023 and FY2024, because those EPS values were rejected in Step 7c.
>   - FY2025 and FY2026 pass the share-basis check (15.0 vs 15.4 bn implied shares). Their prices (₹914, ₹732) are on the post-bonus basis, because Yahoo's `close` is split-adjusted, and so is the EPS.
>   - **Surprise:** ROE falls from 17% in FY2023 to about 9% from FY2024 on. Net income rose 26% across the merger, but stored equity rose 2.4×. HDFC Bank's ROE has historically been in the mid-teens, so this suggests yfinance's "Stockholders Equity" figure for the merged bank may not match the reported net worth. It also drags P/B down, since P/B = P/E × ROE. I haven't changed anything for this: it needs cross-checking against the annual report, not a formula tweak.

---
## M. Checking `update_ratios()` (Step 8b)

Read from the database, calculate with `compute_ratios()`, bulk-update by primary key. Run twice.

In [51]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from sqlalchemy import text

from data.database import SessionLocal, engine
from data.ingestion.ratios import update_ratios

pd.set_option("display.float_format", "{:,.4f}".format)

ANNUAL_SQL = text("""
    SELECT s.symbol, f.period_date, f.roe, f.debt_to_equity, f.pe_ratio, f.pb_ratio
    FROM financial_data f JOIN stocks s ON s.id = f.stock_id
    WHERE f.period_type = 'annual' ORDER BY s.symbol, f.period_date
""")

### M1. First run

In [52]:
with SessionLocal() as s:
    for stock_id in (1, 10):          # TCS, HDFC (HDFC's id checked in psql)
        print(stock_id, update_ratios(s, stock_id))
    s.commit()
with engine.connect() as c:
    first = pd.read_sql(ANNUAL_SQL, c)
first

1 4
10 4


,symbol,period_date,roe,debt_to_equity,pe_ratio,pb_ratio
0,HDFCBANK.NS,2023-03-31,0.1701,NaN,NaN,NaN
1,HDFCBANK.NS,2024-03-31,0.0896,NaN,NaN,NaN
2,HDFCBANK.NS,2025-03-31,0.0877,NaN,20.3949,1.7893
3,HDFCBANK.NS,2026-03-31,0.0863,NaN,15.9902,1.3798
4,TCS.NS,2023-03-31,0.4661,0.0850,27.8314,12.9723
5,TCS.NS,2024-03-31,0.5073,0.0886,30.7936,15.6226
6,TCS.NS,2025-03-31,0.5124,0.0991,26.8735,13.7700
7,TCS.NS,2026-03-31,0.4589,0.1052,17.3436,7.9586


### M2. Second run

In [53]:
with SessionLocal() as s:
    for stock_id in (1, 10):          # TCS, HDFC (HDFC's id checked in psql)
        print(stock_id, update_ratios(s, stock_id))
    s.commit()
with engine.connect() as c:
    second = pd.read_sql(ANNUAL_SQL, c)
    quarterly = pd.read_sql(text(
        "SELECT period_type, count(*) FILTER (WHERE roe IS NOT NULL) AS non_null_roe FROM financial_data GROUP BY 1 ORDER BY 1"
    ), c)
print("identical to first run:", second.equals(first))
quarterly

1 4
10 4
identical to first run: True


,period_type,non_null_roe
0,annual,8
1,quarterly,0


### M3. Stale ratios are overwritten, inside one uncommitted transaction

Null TCS's FY2026 EPS without committing, recalculate in the same session, and check that the P/E and P/B built from that EPS became NULL. Then roll back.

In [54]:
with SessionLocal() as s:
    s.execute(text("UPDATE financial_data SET eps = NULL WHERE stock_id = 1 AND period_type = 'annual' AND period_date = '2026-03-31'"))
    update_ratios(s, 1)   # sees the uncommitted NULL through the same session
    during = pd.read_sql(text(
        "SELECT period_date, eps, roe, pe_ratio, pb_ratio FROM financial_data "
        "WHERE stock_id = 1 AND period_type = 'annual' AND period_date >= '2025-03-31' ORDER BY period_date"
    ), s.connection())
    s.rollback()
during

,period_date,eps,roe,pe_ratio,pb_ratio
0,2025-03-31,134.1900,0.5124,26.8735,13.7700
1,2026-03-31,NaN,0.4589,NaN,NaN


> **Finding:** the stored ratios match section L (rounded to 4 decimal places by `Numeric(12, 4)`), and a second run leaves them identical. Quarterly rows have no ratios. In M3, nulling an EPS inside an open transaction and recalculating turned that year's P/E and P/B to NULL, while ROE (which doesn't use EPS) stayed. Writing None explicitly is what clears a stale ratio, and reading through the same session is what lets it see uncommitted raw values.